# Build an uploadable wildfire spread dataset

This notebook builds the manifest-selected, **no-weather** FIRMS/FEDS weak-label dataset. It does not use the Open-Meteo exports because they lack issued-forecast availability provenance.

The currently retained source archive supports `2026-05-31` through `2026-08-10`. Asking for `2026-05-11` through `2026-08-22` correctly fails until FIRMS, FEDS labels, and terrain coverage are collected and rebuilt.

In [ ]:
from datetime import date
from pathlib import Path

from wildfire_data.candidate_dataset import (
    DEFAULT_MODEL_FEATURE_COLUMNS,
    build_and_store_firms_candidate_dataset,
    export_candidate_dataset_release,
    iter_candidate_examples,
)
from wildfire_data.storage_budget import load_storage_budget

DATA_ROOT = Path('data')
START_DATE = date(2026, 5, 31)
END_DATE = date(2026, 8, 10)
RELEASE_DIRECTORY = Path('releases/wildfire-spread-firms-feds-no-weather-2026-05-31_to_2026-08-10')
POLICY = load_storage_budget('config/storage_budget.json')

## Build the candidate feature view

This reads exactly one completed positive-only training manifest, checks terminal FIRMS coverage, produces cutoff-safe FIRMS and terrain features, and makes target-0 rows only as explicitly named weak-negative proxies. The result is invisible to readers until its completed manifest is atomically published.

In [ ]:
build = build_and_store_firms_candidate_dataset(
    DATA_ROOT,
    storage_budget=POLICY,
    start_date=START_DATE,
    end_date=END_DATE,
    # The defaults are a 2 km FIRMS seed radius and 2,000 weak-negative
    # proxies per FEDS source snapshot. Positives are never capped.
)
print(build.manifest_path)
print({
    'candidate_rows': build.candidate_row_count,
    'supported_positives': build.supported_positive_count,
    'weak_negative_proxies': build.weak_negative_proxy_count,
    'unscored_positives': build.unscored_positive_count,
})

## Export the self-contained upload directory

The release contains `candidate_examples.jsonl.gz`, unscored-positive diagnostics, schema, dataset manifest, file inventory, and SHA-256 checksums. It deliberately excludes raw-provider archives and Open-Meteo visualization exports.

In [ ]:
release = export_candidate_dataset_release(
    DATA_ROOT,
    RELEASE_DIRECTORY,
    candidate_manifest=build.manifest_path,
)
print(release.directory)
print(release.manifest_path)

## Audit before training

Use only `DEFAULT_MODEL_FEATURE_COLUMNS` as model inputs. Do not train on IDs, timestamps, labels, raw-artifact IDs, selection metadata, weather-missingness fields, or `dataset_split`. The existing tabular baseline should group its chronological split by `source_snapshot_time`.

In [ ]:
rows = iter_candidate_examples(DATA_ROOT, manifest_path=build.manifest_path)
target_counts = {0: 0, 1: 0}
split_counts = {}
weather_statuses = set()
rows_missing_model_features = 0
for row in rows:
    target_counts[row['target_newly_burned_12h']] += 1
    split_counts[row['dataset_split']] = split_counts.get(row['dataset_split'], 0) + 1
    weather_statuses.add(row['weather_feature_status'])
    rows_missing_model_features += int(any(name not in row for name in DEFAULT_MODEL_FEATURE_COLUMNS))

assert rows_missing_model_features == 0
assert weather_statuses == {'unavailable-no-issued-forecast-features'}
print('target counts:', target_counts)
print('split counts:', split_counts)
print('model features:', list(DEFAULT_MODEL_FEATURE_COLUMNS))

## Optional: fit the first weak-label baseline

This is deliberately opt-in: target=0 means a FIRMS-seeded proxy, not an observed clear/no-burn cell. It persists a model bundle, feature contract, and later-time holdout metrics outside `data/`. Do not use the resulting score as an operational forecast or independent ground-truth evaluation.

In [ ]:
RUN_BASELINE = False
MODEL_DIRECTORY = Path('models/firms-feds-no-weather-2026-05-31_to_2026-08-10')

if RUN_BASELINE:
    import pandas as pd
    from wildfire_data.tabular_baseline import persist_tabular_baseline, train_tabular_baseline

    training_columns = (
        'anchor_at', 'source_snapshot_time', 'target_newly_burned_12h',
        *DEFAULT_MODEL_FEATURE_COLUMNS,
    )
    frame = pd.DataFrame(
        {column: row[column] for column in training_columns}
        for row in iter_candidate_examples(DATA_ROOT, manifest_path=build.manifest_path)
    )
    baseline = train_tabular_baseline(
        frame,
        target_column='target_newly_burned_12h',
        feature_columns=DEFAULT_MODEL_FEATURE_COLUMNS,
        split_group_column='source_snapshot_time',
    )
    persisted = persist_tabular_baseline(baseline, MODEL_DIRECTORY)
    print(persisted)
    print(baseline.metrics.as_dict())
else:
    print('Set RUN_BASELINE = True only to run the explicitly weak-label baseline.')